# Лабораторная работа №8. ICMP и traceroute: сеть рассказывает о себе

Стенд уже маршрутизируется. Мы будем менять TTL вручную и смотреть, какой router отвечает ICMP Time Exceeded.

## Схема

![TTL](assets/ttl.svg)

Путь: `pc1 → r1 → r2 → pc2`. Каждый router уменьшает IPv4 TTL на единицу.

## 1. Убедимся, что обычная связность есть

На `pc1`:

```bash
ip route get 10.8.2.10
ping -c 2 10.8.2.10
```

Если обычный ping не работает, TTL-эксперимент теряет смысл: сначала должен существовать корректный route path.

### Q1. Зачем IP header вообще содержит TTL, если routing table уже определяет путь?

`введите ваш ответ`

## 2. TTL = 1

На `pc1`:

```bash
ping -c 1 -t 1 10.8.2.10
```

Пакет достигает R1, TTL становится 0, и R1 не пересылает packet дальше. Вместо этого он возвращает ICMP Time Exceeded.

### Q2. Какой source IP вы ожидаете у ICMP Time Exceeded для TTL=1?

`введите ваш ответ`

## 3. TTL = 2 и TTL = 3

```bash
ping -c 1 -t 2 10.8.2.10
ping -c 1 -t 3 10.8.2.10
```

Сравните, кто отвечает. TTL=2 должен «дожить» до R2, TTL=3 — до конечного host.

In [ ]:
%%capture_traffic pc1:eth1 --filter "icmp" --timeout 15 --packets 100 --save captures/ttl-icmp.pcap
import time
print("В terminal pc1 выполните ping с TTL 1, 2 и 3")
time.sleep(13)

In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display
viewer = PcapViewer("captures/ttl-icmp.pcap", "icmp", limit=50)
frames = viewer.packets()
display(frames)
if not frames.empty:
    for n in frames["№"].head(4):
        print("\nframe", n)
        print("\n".join(viewer.packet(int(n), "protocols").splitlines()[:60]))

### Q3. Почему traceroute может узнать последовательность router-ов, не имея доступа к их routing tables?

`введите ваш ответ`

## 4. Теперь traceroute

```bash
traceroute -I 10.8.2.10
```

Сопоставьте hop 1/2/3 с предыдущими ping TTL=1/2/3. Это тот же механизм, только автоматизированный серией probe с растущим TTL.

## CHECK

Checker проверяет, что преднастроенный route path жив и `pc1 → pc2` проходит. TTL/traceroute — наблюдаемая часть лабораторной, а не искусственный state для grading.

## Итог

ICMP — не только Echo. Time Exceeded превращает ограничение TTL в диагностический сигнал, а traceroute систематически собирает эти ответы hop-by-hop.